In [2]:
# CREATE TABLE fact_pit_stop (
#     pit_id bigserial primary key,
#     session_id int not null,
#     driver_id int not null,
#     lap_number int,
#     pit_duration_s numeric,
#     compound_in_id int,
#     compound_out_id int,
#     position_before int,
#     position_after int,

#     CONSTRAINT fk_pit_session      FOREIGN KEY (session_id)      REFERENCES dim_session(session_id),
#     CONSTRAINT fk_pit_driver       FOREIGN KEY (driver_id)       REFERENCES dim_driver(driver_id),
#     CONSTRAINT fk_pit_compound_in  FOREIGN KEY (compound_in_id)  REFERENCES dim_compound(compound_id),
#     CONSTRAINT fk_pit_compound_out FOREIGN KEY (compound_out_id) REFERENCES dim_compound(compound_id)
# );
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import fastf1

fastf1.Cache.enable_cache('/tmp/fastf1_cache')

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 14:31:52 WARN Utils: Your hostname, michal-MS-7996, resolves to a loopback address: 127.0.1.1; using 192.168.0.58 instead (on interface enp2s0)
26/03/31 14:31:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/michal/.ivy2.5.2/cache
The jars for the packages stored in: /home/michal/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a7a18a52-71c3-4eb5-9f21-ab581d71b2eb;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 142ms :: artifacts dl 5ms
	:: modules in use:
	org.checkerfram

In [3]:
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
dim_season_df = dim_season.collect()
dim_session = spark.read.jdbc(url=url, table='dim_session', properties=properties)
dim_session_df = dim_session.collect()
dim_driver = spark.read.jdbc(url=url, table='dim_driver', properties=properties)
dim_driver_df = dim_driver.collect()
dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)
dim_event_df = dim_event.collect()
dim_compound = spark.read.jdbc(url=url, table='dim_compound', properties=properties)
dim_compound_df = dim_compound.collect()

season_map = {row['season_id']: row['year'] for row in dim_season_df}
driver_map = {row['code']: row['driver_id'] for row in dim_driver_df}
event_map = {
    row["event_id"]: (row["season_id"], row["round_number"])
    for row in dim_event.collect()
}
compound_map = {row['name']: row['compound_id'] for row in dim_compound_df}

In [20]:
def build_fact_pit_stop(session, session_id, driver_map, compound_map, spark):
    laps = session.laps

    for col in laps.columns:
        if pd.api.types.is_timedelta64_dtype(laps[col]):
            laps[col] = laps[col].dt.total_seconds()

    # okrążenie gdzie PitInTime istnieje
    pit_laps = laps[laps["PitInTime"].notna()].copy()

    if pit_laps.empty:
        print(f"Brak pit stopów dla session_id={session_id}")
        return None
    
    # compound_in = compound na okrążeniu pit in (stary kompound)
    pit_laps = pit_laps.rename(columns={"Compound": "compound_out"})

    # compound_out = compound na następnym okrążeniu (nowy kompound)
    # PitOutTime istnieje na okrążeniu wyjazdowym
    pit_out_laps = laps[laps["PitOutTime"].notna()][["Driver", "Stint", "Compound"]].copy()
    pit_out_laps = pit_out_laps.rename(columns={"Compound": "compound_in", "Stint": "Stint_out"})

    #stint po pit stopie = stint_in = stint_out w pit_laps + 1
    pit_laps["Stint_out"] = pit_laps["Stint"] + 1
    pit_laps = pit_laps.merge(pit_out_laps, on=["Driver", "Stint_out"], how="left")

    # position_after = pozycja na następnym okrążeniu
    next_lap_pos = laps[["Driver", "LapNumber", "Position"]].copy()
    next_lap_pos["LapNumber"] = next_lap_pos["LapNumber"] - 1  # przesuń o 1
    next_lap_pos = next_lap_pos.rename(columns={"Position": "position_after"})

    pit_laps = pit_laps.rename(columns={"Position": "position_before"})
    pit_laps = pit_laps.merge(
        next_lap_pos,
        on=["Driver", "LapNumber"],
        how="left"
    )

    # pit_duration_s
    # PitOutTime następnego okrążenia - PitInTime tego okrążenia
    pit_out_times = laps[laps["PitOutTime"].notna()][["Driver", "Stint", "PitOutTime"]].copy()
    pit_out_times = pit_out_times.rename(columns={"Stint": "Stint_out", "PitOutTime": "pit_out_s"})
    pit_laps = pit_laps.merge(pit_out_times, on=["Driver", "Stint_out"], how="left")
    pit_laps["pit_duration_s"] = pit_laps["pit_out_s"] - pit_laps["PitInTime"]
    
    pit_laps = pit_laps[["Driver", "LapNumber",
        "pit_duration_s", "compound_in", "compound_out",
        "position_before", "position_after"]]
    
    pit_df = spark.createDataFrame(pit_laps)
    pit_df = pit_df.withColumn("session_id", F.lit(session_id))

    driver_mapping = F.create_map([F.lit(x) for pair in driver_map.items() for x in pair])
    compound_mapping = F.create_map([F.lit(x) for pair in compound_map.items() for x in pair])

    pit_df = pit_df \
        .withColumn("driver_id",       driver_mapping[F.col("Driver")]) \
        .withColumn("compound_in_id",  compound_mapping[F.col("compound_in")]) \
        .withColumn("compound_out_id", compound_mapping[F.col("compound_out")])

    pit_df = pit_df.select([
        "session_id", "driver_id", "LapNumber",
        "pit_duration_s", "compound_in_id", "compound_out_id",
        "position_before", "position_after",
    ])

    pit_df = pit_df.withColumnRenamed("LapNumber", "lap_number")

    # NaN -> null
    float_cols = [f.name for f in pit_df.schema.fields
                  if isinstance(f.dataType, (DoubleType, FloatType))]
    for col in float_cols:
        pit_df = pit_df.withColumn(col,
            F.when(F.isnan(F.col(col)), None).otherwise(F.col(col)))

    pit_df = pit_df \
        .withColumn("driver_id",       F.expr("try_cast(driver_id       as INT)")) \
        .withColumn("compound_in_id",  F.expr("try_cast(compound_in_id  as INT)")) \
        .withColumn("compound_out_id", F.expr("try_cast(compound_out_id as INT)")) \
        .withColumn("lap_number",      F.expr("try_cast(lap_number      as INT)")) \
        .withColumn("position_before", F.expr("try_cast(position_before as INT)")) \
        .withColumn("position_after",  F.expr("try_cast(position_after  as INT)"))

    return pit_df

In [23]:
loaded = 0
errors = 0
for row in dim_session_df:
    season_id, round_number = event_map[row.event_id]

    if row.session_type not in ["R", "S"]:
        continue

    try:
        session = fastf1.get_session(season_map[season_id], round_number, row.session_type)
        session.load(laps=True, telemetry=False, weather=False, messages=False)
    except Exception as e:
        print(f"Failed: {season_map[season_id]} R{round_number} {row.session_type}: {e}")
        loaded += 1
        continue

    sdf = build_fact_pit_stop(session, row.session_id, driver_map, compound_map, spark)
    if sdf:
        sdf.write.jdbc(url=url, table='fact_pit_stop', mode='append', properties=properties)
        errors += 1
    
print(f'loaded {loaded/(loaded+errors)*100}% ({loaded}, {errors})')

Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2024.json failed; using cached response
Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Austrian Grand Prix - Sprint [v3.8.1]
req            INFO 	Usi

Brak pit stopów dla session_id=3


core           INFO 	Finished loading data for 20 drivers: ['63', '81', '55', '44', '1', '27', '11', '20', '3', '10', '16', '31', '18', '22', '23', '77', '24', '14', '2', '4']
Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2024.json failed; using cached response
Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError

Brak pit stopów dla session_id=43


Request for URL https://api.jolpi.ca/ergast/f1/2024/19/results.json failed; using cached response
Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req          

...

Failed: 2024 R9 R: any API: 500 calls/h


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cached response
Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Dutch Grand Prix - Race [v3.8.1]
req            INFO 	No cache

Failed: 2022 R15 R: any API: 500 calls/h


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cached response
Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.1]
req            INFO 	No cac

Failed: 2022 R16 R: any API: 500 calls/h
loaded 46.51162790697674% (80, 92)


In [24]:
#~6400 rekordów